In [7]:
import numpy as np
import OpenVisus as ov
import matplotlib.pyplot as plt
import os
from pathlib import Path
import pandas as pd
import pyvista as pv
from tqdm import tqdm

In [8]:
# === params (temperature output, appended as NEW CELLS) ===
# NOTE: paste this cell directly below your existing pressure params / generation cells.

# Safety imports in case this cell is run standalone
import numpy as np
from pathlib import Path

# Frame / sampling params (match these to your pressure cell if you changed them)
num_frames = 500
face = 2
quality = -8  # 1/8th downsample (adjust if your data source uses a different mapping)
frames = np.linspace(0, 10268, num_frames, dtype=int).tolist()

# Variable name for temperature in your DB. If your DB uses 'temp' or 'temperature', change this.
variables = ['T']

# Output directory & series file for temperature
output_dir = Path(f"temperature_vtk({quality})")
pvd_path = output_dir / "temperature_series.pvd"
output_dir.mkdir(parents=True, exist_ok=True)
# === end params ===

In [9]:
# initialize databases
output_dir.mkdir(exist_ok=True)
times = np.linspace(0, 1, num_frames)

dbs = []
urls = []
for variable in variables:
    geos_face_loc=f"https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/GEOS/GEOS_{variable.upper()}/{variable.lower()}_face_{face}_depth_52_time_0_10269.idx"
    urls.append(geos_face_loc)
    dbs.append(ov.LoadDataset(geos_face_loc))
# Create coordinate indices
data = dbs[0].read(
    time=0,
    quality=quality
)
Z, Y, X = data.shape
grid = pv.ImageData()
grid.dimensions = (X, Y, Z)
grid.origin = (0, 0, 0)
grid.spacing = (1, 1, 20)
print(data.shape)

(7, 180, 360)


In [10]:
# === per-frame reading/writing for TEMPERATURE (NEW CELL) ===
# IMPORTANT: Run this cell only AFTER the notebook has created/initialized:
#   - `dbs` (list/objects used to read variables, same as pressure pipeline)
#   - `grid` (pyvista/meshio/VTK grid object configured as in the pressure pipeline)
#   - any imports used by your pressure pipeline (e.g., numpy as np, Path, pyvista, etc.)
#
# This cell mirrors the pressure loop but writes the scalar field "temperature".

# If your temperature dataset is not at dbs[0], update the index below to the correct dbs[i].
# If your temperature variable name is not 'T', either change `variables` in the params cell
# or adjust the read call here accordingly.

for i, t in enumerate(frames):
    print(f"Writing temperature frame {i}/{len(frames)-1} (time={t})")
    # Read temperature variable (assumes dbs[0] is the temperature dataset)
    T = dbs[0].read(time=t, quality=quality)   # expected shape (Nz, Ny, Nx) or similar

    # Flatten ordering:
    # - Default: C-order flattening (T.flatten()) — keep if your pressure pipeline used C-order
    # - If the result looks transposed or collapsed, use order='F' (Fortran) instead.
    T_flat = T.flatten()         # try this first
    # T_flat = np.ravel(T, order='F')   # alternative if orientation is wrong

    # Assign scalar field named "temperature" into the grid
    grid["temperature"] = T_flat

    # (Optional) If your grid needs explicit spacing/stretching for visible thickness, set:
    # grid.spacing = (1, 1, 15)   # tune the 3rd value to stretch Z visually (uncomment if needed)

    filename = output_dir / f"frame_{i:04d}.vti"
    grid.save(filename)
# === end processing cell ===

Writing temperature frame 0/499 (time=0)
Writing temperature frame 1/499 (time=20)
Writing temperature frame 2/499 (time=41)
Writing temperature frame 3/499 (time=61)
Writing temperature frame 4/499 (time=82)
Writing temperature frame 5/499 (time=102)
Writing temperature frame 6/499 (time=123)
Writing temperature frame 7/499 (time=144)
Writing temperature frame 8/499 (time=164)
Writing temperature frame 9/499 (time=185)
Writing temperature frame 10/499 (time=205)
Writing temperature frame 11/499 (time=226)
Writing temperature frame 12/499 (time=246)
Writing temperature frame 13/499 (time=267)
Writing temperature frame 14/499 (time=288)
Writing temperature frame 15/499 (time=308)
Writing temperature frame 16/499 (time=329)
Writing temperature frame 17/499 (time=349)
Writing temperature frame 18/499 (time=370)
Writing temperature frame 19/499 (time=390)
Writing temperature frame 20/499 (time=411)
Writing temperature frame 21/499 (time=432)
Writing temperature frame 22/499 (time=452)
Writ

In [11]:
files = []
for i, t in enumerate(frames):
    filename = output_dir / f"frame_{i:04d}.vti"
    files.append((i, filename.name))


In [12]:

with open(pvd_path, "w") as f:
    f.write('<?xml version="1.0"?>\n')
    f.write('<VTKFile type="Collection" version="0.1">\n')
    f.write('  <Collection>\n')
    
    for t, fname in tqdm(files, desc="Writing PVD"):
        f.write(f'    <DataSet timestep="{t}" file="{fname}"/>\n')
    
    f.write('  </Collection>\n')
    f.write('</VTKFile>\n')

Writing PVD: 100%|███████████████████████████████████████████████████████████████████████████| 500/500 [00:00<?, ?it/s]


In [14]:
# Read only pressure 'p' (assumes dbs[0] is pressure)
p = dbs[0].read(time=t, quality=quality)
p_flat = p.flatten()
grid["pressure"] = p_flat
filename = output_dir / f"frame_{i:04d}.vti"
grid.save(filename)

In [15]:
    # Read temperature variable (assumes dbs[0] is temperature dataset)
    T = dbs[0].read(time=t, quality=quality)   # shape: (Nz, Ny, Nx) or similar

    # flatten into the 1D array that VTK expects (adjust order if needed)
    T_flat = T.flatten()

    # write the scalar field named "temperature" into the grid
    grid["temperature"] = T_flat

    filename = output_dir / f"frame_{i:04d}.vti"
    grid.save(filename)

In [16]:
T_flat = np.ravel(T, order='F')
# or
T_flat = T.flatten(order='F')

In [17]:
grid.spacing = (1, 1, 20)   # tune the 3rd value until it visually matches expected thickness